# Gold Enriched

Este notebook cria e mantém a tabela **`gold.tb_sales_enriched`** — a camada gold de vendas enriquecida com as características do canal de comercialização.

**Objetivo:**
- Ler as vendas da camada silver (`silver.tb_sales`)
- Enriquecer os dados com um `LEFT JOIN` na tabela de características do canal (`silver.tb_channel_group`), trazendo `trade_group_desc` e `trade_type_desc` por meio de `trade_chnl_desc`
- Gravar o resultado na tabela gold `gold.tb_sales_enriched`, no grão de **1 linha por `sales_key`**

**Fluxo das células:**
1. **Configuração** — define o catálogo via widget `CATALOG` e os nomes das tabelas de origem (silver) e destino (gold)
2. **Criação da tabela** — `CREATE TABLE IF NOT EXISTS` com a estrutura Delta da tabela destino
3. **Carga incremental** — `MERGE INTO` que insere novas chaves e atualiza as existentes (`UPDATE SET *` / `INSERT *`), registrando `_gold_updated_at` a cada execução

**Como usar:** informe o parâmetro `CATALOG` (catálogo do workspace) ao executar o notebook, por exemplo em um job do Databricks.

In [0]:
dbutils.widgets.text("CATALOG", "")

CATALOG = dbutils.widgets.get("CATALOG")

SALES = f"{CATALOG}.silver.tb_sales"
CHANNEL = f"{CATALOG}.silver.tb_channel_group"
TARGET = f"{CATALOG}.gold.tb_sales_enriched"


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET} (
    sales_key            STRING,
    sale_date            DATE,
    ce_brand_flvr        INT,
    brand_nm             STRING,
    btlr_org_lvl_c_desc  STRING,
    chnl_group           STRING,
    trade_chnl_desc      STRING,
    pkg_cat              STRING,
    pkg_cat_desc         STRING,
    tsr_pckg_nm          STRING,
    sales_volume         DECIMAL(18,2),
    year                 INT,
    month                INT,
    period               INT,
    trade_group_desc     STRING,
    trade_type_desc      STRING,
    _source_file         STRING,
    _ingestion_timestamp TIMESTAMP,
    _gold_updated_at     TIMESTAMP
)
USING DELTA
COMMENT 'Vendas (silver) enriquecidas com as características do canal. Grão: 1 linha por sales_key.'
""")

In [0]:
spark.sql(f"""
MERGE INTO {TARGET} AS t
USING (
    SELECT s.sales_key, s.sale_date, s.ce_brand_flvr, s.brand_nm, s.btlr_org_lvl_c_desc, s.chnl_group,
           s.trade_chnl_desc, s.pkg_cat, s.pkg_cat_desc, s.tsr_pckg_nm, s.sales_volume,
           s.year, s.month, s.period,
           c.trade_group_desc, c.trade_type_desc,
           s._source_file, s._ingestion_timestamp,
           current_timestamp() AS _gold_updated_at
    FROM {SALES} s
    LEFT JOIN {CHANNEL} c
           ON s.trade_chnl_desc = c.trade_chnl_desc
) AS s
ON t.sales_key = s.sales_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")